# SpeechT5 + WavLM Fine-Tuning

This notebook fine-tunes **SpeechT5** (`microsoft/speecht5_vc`) to accept **WavLM hidden states**
as encoder input instead of raw audio, enabling a WavLM-based feature-space speech translation pipeline.

## Architecture
```
EN raw audio  ──► [WavLM Encoder (frozen)]  ──► EN hidden states (Seq, 768)
                                                        │
                                 [SpeechT5 Transformer (fine-tuned)] ──► DE mel spectrogram
                                                        │
                                             [HiFi-GAN Vocoder]  ──► DE audio
```

## Required Pre-Step
The preprocessed WavLM dataset must exist at:
```
datasets/processed_wavlm_en_de_v1/
    en/   ← WavLM hidden states (Seq, 768)
    de/   ← WavLM hidden states (Seq, 768)
```
Run `preprocess_wavlm.py` if the dataset is not yet generated.

> **Note on the target representation:**  
> The dataset stores WavLM features for both sides. The `SpeechT5WavLMDataset`
> class in `model.py` handles the fallback to mel-spectrogram extraction if the
> target is raw audio. The fine-tuning loss is computed against SpeechT5's
> mel-spectrogram decoder output — this is what teaches the model to bridge
> the WavLM representation into something the trained decoder can process.


## 1. Setup & Imports

In [ ]:
import os
import sys

# Add project root to Python path so that dataset_loader and encoders are importable.
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, project_root)

from model import SpeechT5WavLM
import dataset_loader
from datasets import load_from_disk
import numpy as np
from IPython.display import Audio, display

print(f"Project root: {project_root}")
print(f"Datasets directory: {dataset_loader.DATASETS_DIR}")

Project root: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model
Datasets directory: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets


## 2. Configuration

Edit the constants below to control training behaviour.

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Path to the preprocessed WavLM dataset produced by preprocess_wavlm.py.
# ─────────────────────────────────────────────────────────────────────────────
PREPROCESSED_PATH = os.path.join(dataset_loader.DATASETS_DIR, "processed_wavlm_en_de_v2")

# ─────────────────────────────────────────────────────────────────────────────
# Training hyper-parameters
# ─────────────────────────────────────────────────────────────────────────────
EPOCHS        = 20       # Total training epochs
BATCH_SIZE    = 1        # Keep small — WavLM sequences are variable length
LEARNING_RATE = 2e-5     # Low LR for stable fine-tuning of pre-trained weights

# ─────────────────────────────────────────────────────────────────────────────
# Output checkpoint directory
# ─────────────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = "speecht5_wavlm_en_de_v3"

print(f"Preprocessed data: {PREPROCESSED_PATH}")
print(f"Epochs: {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE}")
print(f"Checkpoint will be saved to: {CHECKPOINT_DIR}")

Preprocessed data: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_wavlm_en_de_v2
Epochs: 20  |  Batch size: 1  |  LR: 2e-05
Checkpoint will be saved to: speecht5_wavlm_en_de_v3


## 3. Verify Preprocessed Dataset

Confirm the dataset exists and inspect a few sample feature shapes before starting training.

In [ ]:
assert os.path.isdir(PREPROCESSED_PATH), (
    f"Preprocessed dataset not found at '{PREPROCESSED_PATH}'.\n"
    "Run preprocess_wavlm.py first."
)

# Load the unified paired dataset (v2 format)
paired_ds = load_from_disk(PREPROCESSED_PATH)

print(f"\nTotal pairs: {len(paired_ds)}")
print(f"Dataset columns: {paired_ds.column_names}")

print("\nSample feature shapes (first 5 pairs):")
for i in range(min(5, len(paired_ds))):
    src = np.array(paired_ds[i]['input_values'])
    tgt = np.array(paired_ds[i]['labels'])
    
    # Source should be WavLM (Seq, 768)
    if src.ndim == 1 and src.size % 768 == 0:
        src = src.reshape(-1, 768)
    
    # Target should be Mel (T, 80)
    # If this fails, the sample contains raw audio or is corrupted
    if tgt.ndim == 1:
        if tgt.size % 80 == 0:
            tgt = tgt.reshape(-1, 80)
            target_info = f"Mel: {tgt.shape}"
        else:
            target_info = f"INVALID (Size {tgt.size} not divisible by 80. Likely raw audio!)"
    else:
        target_info = f"Shape: {tgt.shape}"
    
    print(f"  [{i}] Source (WavLM): {src.shape if src.ndim > 1 else f'Flat {src.size}'} | Target: {target_info}")


AssertionError: Preprocessed dataset not found at '/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_wavlm_en_de_v2'.
Run preprocess_wavlm.py first.

## 4. Initialise the Model

`SpeechT5WavLM` loads:
- **SpeechT5** (`microsoft/speecht5_vc`) — the transformer to fine-tune
- **HiFi-GAN** (`microsoft/speecht5_hifigan`) — the vocoder (frozen, used at inference)

WavLM itself is **not** loaded here; it was already used during preprocessing to produce the
hidden states stored in the dataset. It is only needed again at inference time
(handled by `run_inference`).

In [4]:
model = SpeechT5WavLM()
model.load("speecht5_wavlm_en_de_v2")
print(f"\nModel loaded on device: {model.device}")

Loading SpeechT5WavLM components (WavLM=microsoft/wavlm-base-plus)...
Using device: cuda

Model loaded on device: cuda


## 5. Extract Target Speaker Embedding

The SpeechT5 decoder needs an **x-vector speaker embedding** to condition its output voice.

`get_speaker_embedding` streams one sample from Google FLEURS for the target language and
computes the x-vector using the SpeechBrain classifier. The embedding is saved alongside
the model checkpoint.

In [5]:
model.get_speaker_embedding(target_lang="de")

Initializing X-Vector classifier for embedding extraction...


/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)


Extracting target speaker embedding...
Embedding extracted successfully.


## 6. Fine-Tune

Training will:
1. Load the preprocessed WavLM dataset.
2. Freeze SpeechT5's CNN feature encoder (only transformer layers are trained).
3. Feed WavLM hidden states directly into SpeechT5's transformer encoder.
4. Compute the spectrogram loss against the decoder's mel output.
5. Save a checkpoint every 5 epochs. On `KeyboardInterrupt` progress is saved
   automatically to `speecht5_wavlm_interrupted`.

In [8]:
model.fine_tune(
    preprocessed_path=PREPROCESSED_PATH,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
)

Starting WavLM+SpeechT5 fine-tuning (Hybrid Architecture).
Loading preprocessed data from /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_wavlm_en_de_v2...
Loaded 13438 aligned (source, target) pairs.
Dataset columns: ['id', 'language', 'gender', 'transcription', 'input_values', 'labels']
Freezing SpeechT5 CNN feature encoder (not used in hybrid path).


Epoch 1/20: 100%|██████████| 13438/13438 [19:18<00:00, 11.60it/s, loss=10.0377, l1_pre=5.1377, l1_post=4.8999]  


Epoch 1/20  Avg Loss: 12.3884


Epoch 2/20:   6%|▋         | 845/13438 [01:13<18:19, 11.45it/s, loss=9.5061, l1_pre=4.8741, l1_post=4.6320] 
/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/transformers/configuration_utils.py:461: UserWarning: Some non-default generation parameters are set in the model config. These should go into either a) `model.generation_config` (as opposed to `model.config`); OR b) a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model).This warning will become an exception in the future.
Non-default generation parameters: {'max_length': 1876}
  warnings.warn(



Training interrupted! Saving current progress...
Saved to 'speecht5_wavlm_interrupted'. Exiting safely.


## 7. Save the Final Checkpoint

In [ ]:
model.save(CHECKPOINT_DIR)
print(f"Model saved to: {CHECKPOINT_DIR}")

## 8. Quick Inference Test

Load a raw EN clip from the raw dataset and run it through the full pipeline
(WavLM → fine-tuned SpeechT5 transformer → HiFi-GAN) to hear if the model is learning.

In [ ]:
source_lang = "en"
target_lang = "de"
# model.load("checkpoint_epoch_20")
# Load a raw EN sample from the seamless_align dataset for inference.
print("Loading a raw EN sample for inference test...")
raw = dataset_loader.load_data(
    lang=[source_lang],
    split="train",
    dataset=["fleurs"],
    num_samples=1000,
)
sample = raw[source_lang][5]
audio_array = np.array(sample['audio']['array'], dtype=np.float32)
sr = sample['audio']['sampling_rate']

print(f"Input audio: {len(audio_array)/sr:.2f}s @ {sr}Hz")
print("Original EN audio:")
display(Audio(data=audio_array, rate=sr))

# Run the full pipeline: raw audio → WavLM → fine-tuned SpeechT5 → HiFi-GAN → audio
print("\nRunning fine-tuned inference...")
result = model.run_inference(
    audio_array=audio_array,
    sampling_rate=sr,
    threshold=0.5,
    minlenratio=0.0,
    maxlenratio=2.0,
)

out_audio = result['audio']['array']
out_sr    = result['audio']['sampling_rate']

print(f"Output audio: {len(out_audio)/out_sr:.2f}s @ {out_sr}Hz")
print("\nGenerated DE audio (after fine-tuning):")
display(Audio(data=out_audio, rate=out_sr))

Loading a raw EN sample for inference test...
Loading google/fleurs (en_us) from local storage: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/fleurs/en_us/train...


Slicing loaded dataset (0:1000)...


Map (num_proc=4): 100%|██████████| 1000/1000 [00:01<00:00, 935.38 examples/s]

Validating en (checking audio & uniqueness)...



Validating en (num_proc=4): 100%|██████████| 1000/1000 [00:00<00:00, 1433.78 examples/s]


  823 valid unique IDs found.
Final Count: 823 common valid samples.
Input audio: 11.88s @ 16000Hz
Original EN audio:



Running fine-tuned inference...
Loading WavLM (microsoft/wavlm-base-plus) extractor for inference...
Output audio: 18.98s @ 16000Hz

Generated DE audio (after fine-tuning):


## Bonus: Resume Training from a Checkpoint

If training was interrupted, load the saved checkpoint and continue.

In [ ]:
# ── Uncomment to resume from a saved checkpoint ──────────────────────────────
# RESUME_CHECKPOINT = "speecht5_wavlm_en_de_v2"
# model = SpeechT5WavLM()
# model.load(RESUME_CHECKPOINT)
# model.fine_tune(
#     preprocessed_path=PREPROCESSED_PATH,
#     epochs=EPOCHS,
#     learning_rate=LEARNING_RATE,
#     batch_size=BATCH_SIZE,
# )

Loading SpeechT5WavLM components (WavLM=microsoft/wavlm-base-plus)...
Using device: cuda
Starting WavLM+SpeechT5 fine-tuning (Hybrid Architecture).
Loading preprocessed data from /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_wavlm_en_de_v2...
Loaded 13438 aligned (source, target) pairs.
Dataset columns: ['id', 'language', 'gender', 'transcription', 'input_values', 'labels']
Freezing SpeechT5 CNN feature encoder (not used in hybrid path).


Epoch 1/20: 100%|██████████| 13438/13438 [19:06<00:00, 11.72it/s, loss=6.2368, l1_pre=3.3518, l1_post=2.8850] 


Epoch 1/20  Avg Loss: 6.7098


Epoch 2/20: 100%|██████████| 13438/13438 [19:08<00:00, 11.70it/s, loss=7.2392, l1_pre=3.8928, l1_post=3.3465] 


Epoch 2/20  Avg Loss: 6.6340


Epoch 3/20:   5%|▌         | 675/13438 [00:58<18:18, 11.62it/s, loss=6.8683, l1_pre=3.6925, l1_post=3.1758] 
/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/transformers/configuration_utils.py:461: UserWarning: Some non-default generation parameters are set in the model config. These should go into either a) `model.generation_config` (as opposed to `model.config`); OR b) a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model).This warning will become an exception in the future.
Non-default generation parameters: {'max_length': 1876}
  warnings.warn(



Training interrupted! Saving current progress...
Saved to 'speecht5_wavlm_interrupted'. Exiting safely.


## Fine-tune the Vocoder (HiFi-GAN)

If you want to improve the audio quality further, you can fine-tune the vocoder using adversarial training. This freezes the SpeechT5 and WavLM backbones and exclusively updates the HiFi-GAN parameters.

In [6]:

VOCODER_CHECKPOINT_DIR = "speecht5_wavlm_vocoder_en_de_v1"

# ─────────────────────────────────────────────────────────────────────────────
# Vocoder Fine-tuning
# ─────────────────────────────────────────────────────────────────────────────

VOCODER_PREPROCESSED_PATH = os.path.join(dataset_loader.DATASETS_DIR, "processed_vocoder_en_de_v1")
VOCODER_EPOCHS = 10
VOCODER_LEARNING_RATE = 2e-5
VOCODER_BATCH_SIZE = 1

model.fine_tune_vocoder(
    preprocessed_path=VOCODER_PREPROCESSED_PATH,
    epochs=VOCODER_EPOCHS,
    learning_rate=VOCODER_LEARNING_RATE,
    batch_size=VOCODER_BATCH_SIZE
)



Starting Vocoder Fine-Tuning...
Freezing SpeechT5 transformer and WavLM backbone...


/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Initializing Vocoder Trainer...


Epoch 1/10:   0%|          | 0/13438 [00:01<?, ?it/s]
Traceback (most recent call last):
  File "/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/models/SpeechT5WavLM/model.py", line 560, in fine_tune_vocoder
    trainer.train(preprocessed_path, epochs, learning_rate, batch_size)
  File "/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/models/SpeechT5WavLM/vocoder_trainer.py", line 393, in train
    _, y_ds_gs, fmap_rs_s, fmap_gs_s = self.msd(y, y_hat)
                                       ^^^^^^^^^^^^^^^^^^
  File "/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/U

Error during vocoder training: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 3.69 GiB of which 20.19 MiB is free. Including non-PyTorch memory, this process has 3.66 GiB memory in use. Of the allocated memory 3.53 GiB is allocated by PyTorch, and 36.82 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Saved to 'speecht5_wavlm_vocoder_error'. Exiting safely.


In [ ]:
model.save(VOCODER_CHECKPOINT_DIR)